# Named entity recognition & sequence labeling — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Token-wise labels with legal transitions over a sentence
The examples show BIO tags, emission scores, transition constraints, Viterbi decoding and span extraction.

In [ ]:
toks='Ada works at OpenAI'.split(); tags=['B-PER','O','O','B-ORG']; vals=[t!='O' for t in tags]
plt.figure(figsize=(6,2)); plt.imshow([vals],aspect='auto',cmap='Greens'); plt.xticks(range(len(toks)),toks); plt.yticks([]); plt.title('BIO tags mark entity spans')
assert tags[0]=='B-PER' and tags[-1]=='B-ORG'

In [ ]:
emit=np.array([[2,0],[0,2],[1,1]],float); probs=softmax(emit,axis=1)
plt.figure(figsize=(4,3)); plt.imshow(probs,cmap='Blues'); plt.colorbar(); plt.title('Per-token emission probabilities')
assert probs[0,0]>probs[0,1]

In [ ]:
trans=np.array([[1,-2],[0,1]],float)
plt.figure(figsize=(4,3)); plt.imshow(trans,cmap='coolwarm'); plt.colorbar(); plt.title('CRF transitions reward legal tag paths')
assert trans[0,1]<trans[1,1]

In [ ]:
emit=np.array([[2,0],[1,1],[0,2]],float); trans=np.array([[0.5,-1],[0,0.5]]); dp=np.zeros_like(emit); dp[0]=emit[0]
for t in range(1,3): dp[t]=emit[t]+np.max(dp[t-1][:,None]+trans,axis=0)
plt.figure(figsize=(4,3)); plt.imshow(dp,cmap='magma'); plt.colorbar(); plt.title('Viterbi dynamic program accumulates best path scores')
assert int(np.argmax(dp[-1]))==1

In [ ]:
tags=['B-PER','I-PER','O','B-ORG']; spans=[]; cur=[]
for i,t in enumerate(tags):
    if t.startswith('B-'):
        if cur: spans.append(cur)
        cur=[i]
    elif t.startswith('I-') and cur: cur.append(i)
    else:
        if cur: spans.append(cur); cur=[]
if cur: spans.append(cur)
plt.figure(figsize=(5,2)); plt.bar(range(len(tags)),[1 if any(i in s for s in spans) else 0 for i in range(len(tags))]); plt.title('Decode BIO tags into spans')
assert spans==[[0,1],[3]]